[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yildiztu/Code-101-Problems-Logistics-SCM/blob/main/Part%20I%20-%20The%20Port%20%26%20Container%20Terminal%20%28Problems%201-46%29/D.%20Integrated%20Tactical%20%26%20Pre-Planning%20Problems%20%28Connecting%20the%20Silos%29/29.%20The%20Integrated%20Quay%20Crane%20%26%20Yard%20Truck%20Scheduling%20Problem/P29-Tier-2_executed.ipynb)

---


# 29. The Integrated Quay Crane and Yard Truck Scheduling Problem

## Tier 2: A Weighted Earliest-Deadline-First Heuristic

### Goal
Build a feasible schedule for container-handling jobs using a priority queue and coordinated quay-crane/yard-truck pairs. The heuristic assigns jobs in priority order to the pair that becomes available first. The pairs may work in parallel, so a job's completion does not prevent another available pair from starting its assigned job.

### Inputs and outputs
The input is a set of containers, each with a deadline, handling duration, and yard location, together with a set of synchronized crane-truck pairs. Yard travel times and priority weights are fixed model parameters in this example. The output is a dispatch order, a pair assignment for every container, start and completion times, and summary measures of schedule performance.

### Model assumptions and scope
- All containers are available for processing at time zero; there are no later job-release times.
- Each container is assigned to exactly one crane-truck pair. A pair handles at most one container at a time, and the crane and truck in that pair are treated as a synchronized resource.
- A pair is occupied for the container's handling time plus its yard travel time. Travel times are deterministic, depend only on the container's yard block, and do not depend on the pair or route history.
- Each pair starts at time zero and is available again when its assigned container's handling and travel are complete. Setup times, equipment failures, congestion, precedence constraints, and interactions between different pairs are outside this example.
- The priority order is calculated once at time zero. It is a weighted, EDF-inspired heuristic rather than a guarantee of globally optimal scheduling or strict EDF ordering in every case.

### Priority rule and scheduling logic
For container $i$, let $d_i$ be its deadline, $h_i$ its handling time, and $t$ the time at which priority is evaluated. The urgency component is the nonnegative time remaining to the deadline. The priority score is

$$
p_i(t) = w_u \max(0, d_i - t) + w_p h_i.
$$

Here, $w_u$ and $w_p$ are the urgency and handling-time weights. The implementation uses $w_u=0.7$ and $w_p=0.3$. Smaller scores have higher dispatch priority, so the min-heap removes the container with the smallest score first. Container ID provides a deterministic tie-break when scores are equal. Since all containers are ranked at time zero, the resulting order is fixed for this schedule; priorities are not recalculated after each assignment.

After choosing the next container, the heuristic assigns it to the pair with the earliest availability. If pair $k$ is available at time $a_k$, and its assigned container has handling time $h_i$ and travel time $r_i$, then

$$
s_i = a_k, \qquad c_i = s_i + h_i + r_i,
$$

where $s_i$ and $c_i$ are the start and completion times. The chosen pair's next availability becomes $c_i$. Repeating this assignment for each container creates parallel schedules on the pairs while preserving the priority-based dispatch order.

### Performance measures
The makespan is the final completion time, $C_{\max}=\max_i c_i$. Deadline adherence is the fraction of containers completed by their deadlines. Average waiting time is the mean start time; because all jobs are available at time zero, each start time is also that container's waiting time. Aggregate resource utilization is total busy time divided by the total pair capacity over the makespan:

$$
U = \frac{\sum_i (h_i+r_i)}{K C_{\max}},
$$

where $K$ is the number of crane-truck pairs. This utilization measure includes idle capacity before the makespan ends, including any unused capacity on a pair that finishes early. A heuristic schedule is feasible under the assumptions above, but the priority rule does not establish that its makespan or deadline performance is optimal.

### Computational effort
For $n$ containers and $K$ pairs, building and removing entries from the priority queue takes $O(n\log n)$ time. Selecting the earliest-available pair by scanning the pair list takes $O(K)$ per container, or $O(nK)$ overall. The total scheduling time is therefore $O(n\log n+nK)$, with $O(n+K)$ storage. The pair scan is retained to make the resource-selection step explicit.

### Example data
The example has five containers with deadlines of 20, 15, 25, 18, and 22 minutes and handling times of 3, 2.5, 4, 3.5, and 2 minutes, respectively. Their yard locations and the two crane-truck pairs are defined in the code below.

In [ ]:
import heapq
import time
from dataclasses import dataclass, field
from math import isclose
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

plt.style.use("default")
sns.set_palette("husl")

In [ ]:
@dataclass
class Container:
    """Container data needed to calculate priority and schedule a job."""

    id: int
    deadline: float
    handling_time: float
    yard_location: int
    priority: float = 0.0
    assigned_crane: Optional[int] = None
    assigned_truck: Optional[int] = None
    start_time: Optional[float] = None
    completion_time: Optional[float] = None


@dataclass
class CraneTruckPair:
    """A synchronized crane-truck resource and its current workload."""

    crane_id: int
    truck_id: int
    available_time: float = 0.0
    total_processing_time: float = 0.0
    containers_processed: List[int] = field(default_factory=list)


@dataclass
class ScheduleEntry:
    """Timing and resource assignment for one scheduled container."""

    container_id: int
    crane_id: int
    truck_id: int
    start_time: float
    completion_time: float
    handling_time: float
    travel_time: float
    priority_at_assignment: float

In [ ]:
class EDFIntegratedScheduler:
    """Construct a weighted EDF-inspired schedule for crane-truck pairs."""

    def __init__(
        self,
        containers: List[Container],
        crane_truck_pairs: List[CraneTruckPair],
        urgency_weight: float = 0.7,
        processing_weight: float = 0.3,
    ):
        if not containers:
            raise ValueError("At least one container is required.")
        if not crane_truck_pairs:
            raise ValueError("At least one crane-truck pair is required.")
        if urgency_weight < 0 or processing_weight < 0:
            raise ValueError("Priority weights must be nonnegative.")
        if not isclose(urgency_weight + processing_weight, 1.0):
            raise ValueError("Priority weights must sum to 1.0.")
        if len({container.id for container in containers}) != len(containers):
            raise ValueError("Container IDs must be unique.")
        if any(container.handling_time < 0 for container in containers):
            raise ValueError("Handling times must be nonnegative.")

        self.containers = containers
        self.pairs = crane_truck_pairs
        self.schedule: List[ScheduleEntry] = []
        self.urgency_weight = urgency_weight
        self.processing_weight = processing_weight
        self.current_time = 0.0

        # Travel duration is determined by the destination yard block.
        self.travel_times: Dict[int, float] = {
            1: 4.0,
            2: 3.5,
            3: 5.0,
            4: 4.5,
        }
        unknown_locations = {
            container.yard_location
            for container in containers
            if container.yard_location not in self.travel_times
        }
        if unknown_locations:
            raise ValueError(
                f"No travel time is defined for yard locations: {sorted(unknown_locations)}"
            )

    def calculate_priority(self, container: Container, current_time: float) -> float:
        """Return a score for which a smaller value means earlier dispatch."""
        time_to_deadline = max(0.0, container.deadline - current_time)
        return (
            self.urgency_weight * time_to_deadline
            + self.processing_weight * container.handling_time
        )

    def get_travel_time(self, yard_location: int) -> float:
        """Return the modeled travel duration for a yard block."""
        return self.travel_times[yard_location]

    def select_best_pair(self, current_time: float) -> CraneTruckPair:
        """Choose the pair with the earliest availability."""
        return min(self.pairs, key=lambda pair: pair.available_time)

    def schedule_containers(self) -> List[ScheduleEntry]:
        """Dispatch containers by priority and assign each to an available pair."""
        if self.schedule:
            raise RuntimeError("A scheduler instance can create only one schedule.")

        priority_queue: List[Tuple[float, int, Container]] = []
        for container in self.containers:
            container.priority = self.calculate_priority(
                container, self.current_time
            )
            # A min-heap makes the lowest weighted deadline/handling score dispatch first.
            heapq.heappush(
                priority_queue,
                (container.priority, container.id, container),
            )

        while priority_queue:
            _, _, container = heapq.heappop(priority_queue)
            best_pair = self.select_best_pair(self.current_time)

            travel_time = self.get_travel_time(container.yard_location)
            start_time = max(self.current_time, best_pair.available_time)
            completion_time = start_time + container.handling_time + travel_time

            entry = ScheduleEntry(
                container_id=container.id,
                crane_id=best_pair.crane_id,
                truck_id=best_pair.truck_id,
                start_time=start_time,
                completion_time=completion_time,
                handling_time=container.handling_time,
                travel_time=travel_time,
                priority_at_assignment=container.priority,
            )
            self.schedule.append(entry)

            container.assigned_crane = best_pair.crane_id
            container.assigned_truck = best_pair.truck_id
            container.start_time = start_time
            container.completion_time = completion_time

            # Pair availability allows another pair to work in parallel.
            best_pair.available_time = completion_time
            best_pair.total_processing_time += container.handling_time + travel_time
            best_pair.containers_processed.append(container.id)

        return self.schedule

    def calculate_performance_metrics(self) -> Dict[str, float]:
        """Calculate schedule-level timing, deadline, and utilization measures."""
        if not self.schedule:
            return {}

        makespan = max(entry.completion_time for entry in self.schedule)
        total_busy_time = sum(
            pair.total_processing_time for pair in self.pairs
        )
        total_pair_capacity = len(self.pairs) * makespan
        average_utilization = (
            total_busy_time / total_pair_capacity
            if total_pair_capacity > 0
            else 0.0
        )

        deadlines = {container.id: container.deadline for container in self.containers}
        on_time = sum(
            entry.completion_time <= deadlines[entry.container_id]
            for entry in self.schedule
        )
        deadline_adherence = on_time / len(self.containers)
        average_waiting_time = float(
            np.mean([entry.start_time for entry in self.schedule])
        )

        return {
            "makespan": makespan,
            "avg_utilization": average_utilization,
            "deadline_adherence": deadline_adherence,
            "avg_waiting_time": average_waiting_time,
            "total_containers": len(self.schedule),
        }

In [ ]:
def create_concrete_example():
    """Create the five-container example and its two resource pairs."""
    containers = [
        Container(1, deadline=20, handling_time=3.0, yard_location=1),
        Container(2, deadline=15, handling_time=2.5, yard_location=2),
        Container(3, deadline=25, handling_time=4.0, yard_location=1),
        Container(4, deadline=18, handling_time=3.5, yard_location=3),
        Container(5, deadline=22, handling_time=2.0, yard_location=2),
    ]
    crane_truck_pairs = [
        CraneTruckPair(crane_id=1, truck_id=1),
        CraneTruckPair(crane_id=2, truck_id=2),
    ]
    return containers, crane_truck_pairs


containers, pairs = create_concrete_example()
print(f"Problem created: {len(containers)} containers, {len(pairs)} crane-truck pairs")
print("\nContainer details:")
for container in containers:
    print(
        f"  Container {container.id}: deadline={container.deadline} min, "
        f"handling time={container.handling_time} min, "
        f"yard block={container.yard_location}"
    )

In [ ]:
def run_edf_scheduler(containers, pairs):
    """Run the priority-queue scheduler and return its results."""
    scheduler = EDFIntegratedScheduler(containers, pairs)

    start_clock = time.perf_counter()
    schedule = scheduler.schedule_containers()
    computation_time = time.perf_counter() - start_clock
    metrics = scheduler.calculate_performance_metrics()

    return scheduler, schedule, metrics, computation_time


scheduler, schedule, metrics, computation_time = run_edf_scheduler(
    containers, pairs
)

print("\n=== WEIGHTED EDF SCHEDULING RESULTS ===")
print(f"Computation time: {computation_time:.6f} seconds")
print(f"Makespan: {metrics['makespan']:.2f} minutes")
print(f"Aggregate resource utilization: {metrics['avg_utilization']:.2%}")
print(f"Deadline adherence: {metrics['deadline_adherence']:.2%}")
print(f"Average waiting time: {metrics['avg_waiting_time']:.2f} minutes")

print("\n=== DISPATCH ORDER AND ASSIGNMENTS ===")
for entry in schedule:
    print(
        f"Container {entry.container_id} -> "
        f"QC{entry.crane_id}-T{entry.truck_id}: "
        f"{entry.start_time:.1f}-{entry.completion_time:.1f} min "
        f"(priority score: {entry.priority_at_assignment:.2f})"
    )

In [ ]:
def visualize_edf_results(scheduler, schedule, metrics, computation_time):
    """Plot the schedule, priority scores, utilization, and summary measures."""
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    fig.suptitle(
        "Weighted EDF Integrated Crane-Truck Scheduling Results",
        fontsize=16,
        fontweight="bold",
    )

    pair_names = [
        f"QC{pair.crane_id}-T{pair.truck_id}" for pair in scheduler.pairs
    ]
    pair_positions = {
        (pair.crane_id, pair.truck_id): position
        for position, pair in enumerate(scheduler.pairs)
    }

    # Each bar covers both handling and travel, matching the modeled pair occupancy.
    ax1 = axes[0, 0]
    colors = plt.get_cmap("tab10")
    for entry in schedule:
        position = pair_positions[(entry.crane_id, entry.truck_id)]
        duration = entry.completion_time - entry.start_time
        ax1.barh(
            position,
            duration,
            left=entry.start_time,
            height=0.45,
            color=colors((entry.container_id - 1) % 10),
            alpha=0.85,
            label=f"Container {entry.container_id}",
        )
        ax1.text(
            entry.start_time + duration / 2,
            position,
            f"C{entry.container_id}",
            ha="center",
            va="center",
            fontsize=9,
            fontweight="bold",
        )
    ax1.set_xlabel("Time (minutes)")
    ax1.set_ylabel("Crane-truck pair")
    ax1.set_title("Schedule Timeline (Gantt Chart)")
    ax1.set_yticks(range(len(pair_names)))
    ax1.set_yticklabels(pair_names)
    ax1.grid(True, axis="x", alpha=0.3)
    handles, labels = ax1.get_legend_handles_labels()
    unique_labels = dict(zip(labels, handles))
    ax1.legend(
        unique_labels.values(),
        unique_labels.keys(),
        bbox_to_anchor=(1.02, 1),
        loc="upper left",
        title="Assigned job",
    )

    ax2 = axes[0, 1]
    container_ids = [entry.container_id for entry in schedule]
    priorities = [entry.priority_at_assignment for entry in schedule]
    handling_times = [entry.handling_time for entry in schedule]
    scatter = ax2.scatter(
        priorities,
        handling_times,
        c=container_ids,
        cmap="viridis",
        s=100,
        alpha=0.8,
    )
    for entry in schedule:
        ax2.annotate(
            f"C{entry.container_id}",
            (entry.priority_at_assignment, entry.handling_time),
            xytext=(5, 5),
            textcoords="offset points",
            fontsize=9,
        )
    ax2.set_xlabel("Priority score (lower is earlier)")
    ax2.set_ylabel("Handling time (minutes)")
    ax2.set_title("Priority Score and Handling Time")
    ax2.grid(True, alpha=0.3)
    colorbar = fig.colorbar(scatter, ax=ax2)
    colorbar.set_label("Container ID")

    ax3 = axes[1, 0]
    makespan = metrics["makespan"]
    pair_utilizations = [
        pair.total_processing_time / makespan if makespan > 0 else 0.0
        for pair in scheduler.pairs
    ]
    bars = ax3.bar(
        pair_names,
        [utilization * 100 for utilization in pair_utilizations],
        color=[colors(position % 10) for position in range(len(pair_names))],
        alpha=0.85,
    )
    ax3.set_xlabel("Crane-truck pair")
    ax3.set_ylabel("Busy time as a share of makespan (%)")
    ax3.set_title("Pair Utilization Over the Schedule Makespan")
    ax3.set_ylim(0, 100)
    ax3.grid(True, axis="y", alpha=0.3)
    for bar, utilization in zip(bars, pair_utilizations):
        ax3.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 1,
            f"{utilization * 100:.1f}%",
            ha="center",
            va="bottom",
            fontweight="bold",
        )

    ax4 = axes[1, 1]
    ax4.axis("off")
    metrics_text = (
        "SCHEDULE PERFORMANCE\n"
        "====================\n\n"
        f"Makespan: {metrics['makespan']:.2f} minutes\n"
        f"Aggregate utilization: {metrics['avg_utilization']:.1%}\n"
        f"Deadline adherence: {metrics['deadline_adherence']:.1%}\n"
        f"Average waiting time: {metrics['avg_waiting_time']:.2f} minutes\n"
        f"Containers scheduled: {metrics['total_containers']}\n\n"
        "Priority-queue work: O(n log n)\n"
        "Pair selection work: O(nK)\n"
        f"Measured runtime: {computation_time:.6f} seconds\n\n"
        "Pairs can operate in parallel; each pair\n"
        "handles one assigned container at a time."
    )
    ax4.text(
        0.05,
        0.95,
        metrics_text,
        transform=ax4.transAxes,
        fontsize=11,
        verticalalignment="top",
        fontfamily="monospace",
        bbox={"boxstyle": "round", "facecolor": "lightblue", "alpha": 0.8},
    )

    plt.tight_layout()
    plt.show()


visualize_edf_results(scheduler, schedule, metrics, computation_time)

In [ ]:
def audit_schedule(scheduler, schedule):
    """Check assignment coverage, processing durations, and pair non-overlap."""
    expected_ids = {container.id for container in scheduler.containers}
    scheduled_ids = [entry.container_id for entry in schedule]
    assert len(scheduled_ids) == len(expected_ids)
    assert set(scheduled_ids) == expected_ids

    for entry in schedule:
        expected_duration = entry.handling_time + entry.travel_time
        actual_duration = entry.completion_time - entry.start_time
        assert isclose(actual_duration, expected_duration, rel_tol=1e-9, abs_tol=1e-9)
        assert entry.start_time >= 0
        assert entry.completion_time >= entry.start_time

    pair_schedules = {}
    for entry in schedule:
        key = (entry.crane_id, entry.truck_id)
        pair_schedules.setdefault(key, []).append(entry)

    for entries in pair_schedules.values():
        entries.sort(key=lambda entry: entry.start_time)
        for previous, following in zip(entries, entries[1:]):
            assert previous.completion_time <= following.start_time

    deadline_by_id = {
        container.id: container.deadline for container in scheduler.containers
    }
    audit_rows = [
        {
            "Container": entry.container_id,
            "Priority score": entry.priority_at_assignment,
            "Pair": f"QC{entry.crane_id}-T{entry.truck_id}",
            "Start (min)": entry.start_time,
            "Completion (min)": entry.completion_time,
            "Deadline (min)": deadline_by_id[entry.container_id],
            "On time": entry.completion_time <= deadline_by_id[entry.container_id],
        }
        for entry in schedule
    ]
    audit_table = pd.DataFrame(audit_rows)
    print("Schedule audit passed: every container is assigned once, durations match, and pairs do not overlap.")
    return audit_table


schedule_audit = audit_schedule(scheduler, schedule)
schedule_audit

In [ ]:
def scalability_analysis():
    """Measure runtime and schedule metrics on reproducible generated instances."""
    print("\n=== SCALABILITY ANALYSIS ===")
    problem_sizes = [5, 10, 15, 20, 25]
    results = []

    for size in problem_sizes:
        # Resetting the seed at each size gives reproducible nested random draws.
        np.random.seed(42)
        test_containers = [
            Container(
                id=index + 1,
                deadline=np.random.uniform(10, 50),
                handling_time=np.random.uniform(2, 6),
                yard_location=int(np.random.randint(1, 5)),
            )
            for index in range(size)
        ]

        num_pairs = max(2, size // 2)
        test_pairs = [
            CraneTruckPair(crane_id=index + 1, truck_id=index + 1)
            for index in range(num_pairs)
        ]

        test_scheduler = EDFIntegratedScheduler(test_containers, test_pairs)
        start_clock = time.perf_counter()
        test_schedule = test_scheduler.schedule_containers()
        elapsed_time = time.perf_counter() - start_clock
        test_metrics = test_scheduler.calculate_performance_metrics()

        results.append(
            {
                "problem_size": size,
                "num_pairs": num_pairs,
                "computation_time": elapsed_time,
                "makespan": test_metrics["makespan"],
                "utilization": test_metrics["avg_utilization"],
                "scheduled_count": len(test_schedule),
            }
        )

    print("\nScalability results:")
    print("Size\tPairs\tRuntime (s)\tMakespan (min)\tUtilization")
    print("-" * 68)
    for result in results:
        print(
            f"{result['problem_size']}\t{result['num_pairs']}\t"
            f"{result['computation_time']:.6f}\t"
            f"{result['makespan']:.2f}\t\t"
            f"{result['utilization']:.2%}"
        )

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    sizes = [result["problem_size"] for result in results]
    runtimes = [result["computation_time"] for result in results]
    utilizations = [result["utilization"] for result in results]

    ax1.plot(sizes, runtimes, "o-", color="#FF6B6B", linewidth=2, markersize=8)
    ax1.set_xlabel("Problem size (containers)")
    ax1.set_ylabel("Measured runtime (seconds)")
    ax1.set_title("Runtime Across Generated Instance Sizes")
    ax1.grid(True, alpha=0.3)

    ax2.plot(
        sizes,
        [utilization * 100 for utilization in utilizations],
        "s-",
        color="#4ECDC4",
        linewidth=2,
        markersize=8,
    )
    ax2.set_xlabel("Problem size (containers)")
    ax2.set_ylabel("Aggregate resource utilization (%)")
    ax2.set_title("Utilization Across Generated Instance Sizes")
    ax2.set_ylim(0, 100)
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()
    return results


scalability_results = scalability_analysis()

## Interpreting the heuristic

The priority queue determines the order in which containers are assigned, while earliest-availability selection determines which crane-truck pair receives each container. These are distinct decisions: priority controls dispatch order; pair availability controls resource assignment. Because every pair has its own availability time, assignments can overlap in time across different pairs even though the dispatch decisions are made one at a time.

The weighted score balances time remaining to a deadline against handling duration. A lower score is selected first. Consequently, this is an EDF-inspired priority rule, not strict deadline sorting: the handling-time term can change the order of containers with different deadlines. The score is calculated at time zero and is not updated during dispatch.

The schedule audit checks that every input container appears exactly once, that each scheduled interval equals its handling time plus its yard travel time, and that a pair is not assigned overlapping jobs. Deadline adherence is reported as a measure of schedule performance, not as a hard constraint: the heuristic can produce late completions.

### Strengths and limitations

**Strengths**
- The heap gives a transparent priority-based dispatch order.
- Assigning each job to the earliest-available pair supports parallel work.
- The method is straightforward to implement and its decisions can be inspected in the schedule table and plots.
- The supplied scalability experiment measures runtime and schedule metrics on reproducibly generated instances.

**Limitations**
- The heuristic does not guarantee an optimal makespan or maximum deadline adherence.
- The weights influence the dispatch order and are fixed for this example.
- Travel time is modeled as a fixed duration for each yard block; route interactions and congestion are not represented.
- The model omits release times, precedence constraints, setups, disruptions, and other operational restrictions.

### Computational interpretation

The priority queue requires $O(n\log n)$ work to insert and remove the $n$ containers. For each container, the implementation scans the $K$ pairs to find the earliest available one, adding $O(nK)$ work. Thus the total scheduling time is $O(n\log n+nK)$, and the memory requirement is $O(n+K)$. The measured runtime cells report observations for the generated instances; they are not universal performance guarantees.

### Further modeling scope

Natural extensions include representing additional operational constraints, using pair- and route-dependent travel durations, or testing other instance sizes within the same scheduling framework. Such extensions require explicit assumptions and data; the present schedule should be interpreted only under the model stated above.